# Receipt extraction: fine-tune Qwen3.5-4B

Fine-tune **Qwen3.5-4B** on the same 200-row golden dataset used for this task's GEPA prompt-optimization runs (`../../gepa_optimize.py`), then evaluate with the **same metric on the same held-out split**, so results are directly comparable.

The teacher scores baked into the final cell (GPT-4o-mini: 72.92 baseline, 84.17 GEPA-optimized; GPT-5.4-mini control run: 72.92 baseline, 79.58 GEPA-optimized) come from the published runs in this task's README. Do not publish placeholder scores.

**Go criterion:** fine-tuned SLM matches or beats the teacher baseline prompt score.

Runtime: Colab **T4 (free)** works for 4B in 4-bit; A100 (Colab Pro) is faster. Runtime -> Change runtime type -> GPU.

In [ ]:
# 1. Install (Colab)
!pip install -q unsloth


In [ ]:
# 2. Fetch the same golden dataset (no upload needed) and replicate the GEPA run's exact split
import json
import random
import urllib.request

SEED = 42
ROW_COUNT = 200
SOURCE_URL = "https://huggingface.co/datasets/Voxel51/scanned_receipts/resolve/main/samples.json"

def input_text(sample):
    detections = sample["text_detections"]["detections"]
    lines = [d["label"].strip() for d in detections if d.get("label", "").strip()]
    return (
        "Extract receipt fields as a JSON object with keys company, address, date, total.\n\n"
        "Receipt OCR text:\n" + "\n".join(lines)
    )

def output_json(sample):
    return json.dumps(
        {
            "company": sample["company"],
            "address": sample["address"],
            "date": sample["date"],
            "total": sample["total"],
        },
        ensure_ascii=False,
        sort_keys=True,
    )

with urllib.request.urlopen(SOURCE_URL, timeout=60) as resp:
    data = json.load(resp)

rows = [
    {"text": input_text(sample), "expected": output_json(sample)}
    for sample in data["samples"]
    if sample.get("text_detections", {}).get("detections")
    and all(sample.get(field) for field in ("company", "address", "date", "total"))
]

if len(rows) < ROW_COUNT:
    raise RuntimeError(f"expected at least {ROW_COUNT} rows, got {len(rows)}")

# CRITICAL: same sample seed and same split shuffle as prepare_data.py + gepa_optimize.py
# in this task folder, so the held-out rows are identical across runs.
rows = random.Random(SEED).sample(rows, ROW_COUNT)
random.Random(SEED).shuffle(rows)
split = int(len(rows) * 0.7)
trainset, devset = rows[:split], rows[split:]
print(f"train {len(trainset)}, held-out {len(devset)}")

In [ ]:
# 3. Same Tier-1 metric as the GEPA spike (pure-python copy of json_field_metric)
def json_field_f1(expected_str: str, actual_str: str) -> float:
    try:
        expected = json.loads(expected_str)
        actual = json.loads(actual_str)
    except json.JSONDecodeError:
        return 0.0
    if not isinstance(expected, dict) or not isinstance(actual, dict):
        return 0.0
    tp = sum(1 for k, v in expected.items() if actual.get(k) == v)
    precision = tp / len(actual) if actual else 0.0
    recall = tp / len(expected) if expected else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


In [ ]:
# 4. Load Qwen3.5-4B in 4-bit + LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-4B",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


In [ ]:
# 5. Eval helper + pre-finetune baseline of the raw SLM on the held-out split
import re

def extract_json(s: str) -> str:
    # drop reasoning block if present, strip code fences, take first {...} block
    s = re.sub(r"<think>.*?(</think>|$)", "", s, flags=re.DOTALL)
    s = re.sub(r"```(json)?", "", s)
    m = re.search(r"\{.*\}", s, re.DOTALL)
    return m.group(0) if m else s

def evaluate_slm(model, tokenizer, devset) -> float:
    FastLanguageModel.for_inference(model)
    scores = []
    for row in devset:
        messages = [{"role": "user", "content": row["text"]}]
        try:  # disable thinking mode if the template supports it
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
        # text= keyword is REQUIRED: Qwen3.5 ships a multimodal Processor,
        # and a positional arg is interpreted as an image
        inputs = tokenizer(text=prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        scores.append(json_field_f1(row["expected"], extract_json(completion)))
    return 100 * sum(scores) / len(scores)

raw_slm_score = evaluate_slm(model, tokenizer, devset)
print(f"Qwen3.5-4B raw (no fine-tune): {raw_slm_score:.2f}")


In [ ]:
# 6. Build chat-format training data and fine-tune (SFT)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_text(row):
    messages = [
        {"role": "user", "content": row["text"]},
        {"role": "assistant", "content": row["expected"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list([to_text(r) for r in trainset])

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# 7. Post-finetune eval on the SAME held-out split: the comparison that matters
finetuned_score = evaluate_slm(model, tokenizer, devset)

# Measured 2026-07-06 by the published GEPA runs (see this task's README).
gpt4o_mini_baseline = 72.92
gpt4o_mini_optimized = 84.17
gpt54_mini_baseline = 72.92
gpt54_mini_optimized = 79.58

print("=" * 52)
print(f"gpt-4o-mini baseline prompt  : {gpt4o_mini_baseline:.2f}")
print(f"gpt-4o-mini GEPA-optimized   : {gpt4o_mini_optimized:.2f}")
print(f"gpt-5.4-mini baseline prompt : {gpt54_mini_baseline:.2f}")
print(f"gpt-5.4-mini GEPA-optimized  : {gpt54_mini_optimized:.2f}")
print(f"Qwen3.5-4B raw               : {raw_slm_score:.2f}")
print(f"Qwen3.5-4B fine-tuned        : {finetuned_score:.2f}")
print("=" * 52)
print("GO criterion: fine-tuned SLM >= teacher baseline prompt score")

In [ ]:
# 8. Persist the adapter (plan: artifacts go to a private HF Hub repo, not Drive)
model.save_pretrained("apprentice-qwen35-4b-lora")
tokenizer.save_pretrained("apprentice-qwen35-4b-lora")

# Optional: push to your private HF Hub repo (needs HF token with write access):
# from huggingface_hub import login; login()
# model.push_to_hub("YOUR_HF_USERNAME/apprentice-qwen35-4b-lora", private=True)
print("saved -> apprentice-qwen35-4b-lora/")


## After this run

1. Publish the two SLM numbers exactly as printed — this repo never carries a number that was not measured.
2. Cost note: Qwen3.5-4B self-hosted costs less per token than gpt-4o-mini-class APIs, with lower latency.
3. If APIs error: Unsloth/TRL move fast. Cross-check against the current official Unsloth Qwen notebook.